In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *
from analysis_village.cc1pi.TLExtensionMethod.ConvolutionTestUtils import *
from analysis_village.cc1pi.TLExtensionMethod.PlottingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
import os, glob

base_path = "."  # match your existing base_path

# Remove every ACLiC-generated artifact for both classes
patterns = [
    "PhysdEdx_cpp*", "Hypfit_cpp*",
    "PhysdEdx_ACLiC_dict*", "Hypfit_ACLiC_dict*",
    "*.pcm", "*.d",
]
for pat in patterns:
    for f in glob.glob(os.path.join(base_path, pat)):
        print("removing", f)
        os.remove(f)

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
import os
from pathlib import Path

PARTICLE = "pion"                                   # "pion" or "muon"
PDG = {"pion": 211, "muon": 13}[PARTICLE]

CAF_DIR = Path("/exp/sbnd/data/users/lpelegri/TLECafpyanaData_gen1")
INPUT_FILES = {
    "muon": {"MC": "TLE_1e20_muon_update_calo.df",
             "Data": "TLE_5e18_muon_data_update_calo.df"},
    "pion": {"MC": "TLE_1e20_pion_stopping_update_calo.df",
             "Data": "TLE_1e20_pion_data_update_calo.df"},
}
LOAD_NFILES = {"MC": 30, "Data": 100}               # last argument of load_df
KEYS2LOAD = ["pfp", "hdr", "histpotdf", "hit0", "hit1", "hit2"]

PHYS_DIR = Path("/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod")
TLE_DIR = Path("/exp/sbnd/data/users/lpelegri/TLE_processed_files")   # existing MC variation outputs

# ---- dE/dx slice fits and parametrizations ----
SLICE_RR_MAX = {"muon": 350.0, "pion": 80.0}       # upper residual range of the slice fits [cm]
SLICE_SIGMA_ERR_RANGE = (1e-4, 0.2)                # slices kept: gsigma_err inside this range
SLICE_PITCH = 0.32                                 # [cm], same pitch as the convolution maps
PION_SHIFT_FROM_MUONS = True                       # pions: beyond PION_RR_JOIN use muon slices at equal βγ
PION_RR_JOIN = 80.0                                # [cm]
PION_OVERLAP = (40.0, 80.0)                        # pion residual range used to match the muon offset [cm]
PARAM_TPC = -1                                     # TPC entry loaded into the per-plane C++ maps

# ---- Track processing ----
CHECK_PUR_COMP = False
MAKE_ENERGY_CUT = False
USE_BEST_PLANE = True
SELECTED_PLANE = 2
SUBTRACT_HITS = True
N_SKIP = 10 if PDG == 211 else 25
N_CORES = max(1, int(os.cpu_count() * 0.8))
MAX_EXTENSION = 300.0   # [cm], only used if Hypfit does not expose max_additional_res_length_pion
MAX_RR_CAP = 800        # [cm], upper limit on the length of the convolution maps

# Column names in the pfp DataFrame
COLS = {
    "true_P": "true_P",
    "range_P": "range_P",
    "length": "length",
    "purity": "purity",
    "completeness": "completeness",
    "PDG": "PDG",
    "end_process": "end_process_string",
    "best_plane": "best_plane",
}

PRESETS = [
    {"preset_key": "range", "method": "range",
     "cleaning_method": "skip_only", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "original", "method": "likelihood",
     "cleaning_method": "skip_only_truncate_harsh", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "normalize_clean_dEdx_include_small", "method": "likelihood",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_og", "method": "convolution_og",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_shifted", "method": "convolution_shifted",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
]
SHIFTED_KEY = "convolution_normalize_clean_dEdx_include_small_shifted"
OG_KEY = "convolution_normalize_clean_dEdx_include_small_og"

# ---- Tests ----
USE_RANGE_MOMENTUM = True          # reference momentum: range_P (True) or true_P (False)
HITS_TO_PLOT = "percent"           # "percent", "used" or "removed"
HIT_BINS_PERCENT = [0, 0.25, 0.5, 0.65, 0.8, 0.9]
HIT_BINS_COUNTS = [0, 50, 100, 200, 300, 500]

FRAC_MAX = 0.25     # tail test: upper edge of the lowest track-fraction bin
RES_MIN = 0.3       # tail test: residual above which a track counts as tail

In [ ]:
# ======================================================================
# Physics classes (PhysdEdx, Hypfit)
# ======================================================================
def load_physics_classes(base_dir=PHYS_DIR):
    """Compiles (if needed) and loads PhysdEdx and Hypfit; returns a Hypfit instance.
    Compiler messages go to the terminal running the Jupyter kernel, not to the cell."""
    for src in ("PhysdEdx.cpp", "Hypfit.cpp"):
        if ROOT.gSystem.CompileMacro(str(base_dir / src), "k") != 1:
            raise RuntimeError(f"Compilation of {src} failed; see the terminal running the kernel")
    return ROOT.Hypfit()


h_fit = load_physics_classes()
PDG_MASS = {13: 105.6583755, 211: 139.57039, 2212: 938.27208816}

In [ ]:
# ======================================================================
# Input samples (cafpyana DataFrames)
# ======================================================================
_LOADED = {}


def get_sample(particle, sample):
    """pfp DataFrame and [hit0, hit1, hit2] DataFrames of one input file, loaded once."""
    key = (particle, sample)
    if key not in _LOADED:
        path = CAF_DIR / INPUT_FILES[particle][sample]
        print(f"Loading {path}")
        dfs = load_df(str(path), KEYS2LOAD, LOAD_NFILES[sample])
        _LOADED[key] = {"pfp": dfs["pfp"], "hits": [dfs["hit0"], dfs["hit1"], dfs["hit2"]]}
    return _LOADED[key]


def drop_sample(particle, sample):
    """Frees the memory of a loaded sample."""
    _LOADED.pop((particle, sample), None)
    gc.collect()

In [ ]:
# ======================================================================
# dE/dx slice fits: theoretical PDF (x) Gaussian(sigma_G), shifted by Delta
# ======================================================================
def build_theoretical_pdf(pdg, rr_center, pitch=SLICE_PITCH):
    """TF1 of the theoretical dE/dx PDF at residual range rr_center [cm]
    (same construction as Hypfit::get_conv_function_map)."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    ke = phys.KEFromRangeSpline(float(rr_center))
    gamma = ke / mass + 1.0
    beta2 = 1.0 - 1.0 / gamma**2
    xi = phys.Landau_xi(ke, pitch)
    kappa = xi / phys.Get_Wmax(ke)
    pdf = ROOT.TF1("", ROOT.PhysdEdx.dEdx_PDF_function, -10.0, 20.0, 5)
    pdf.SetParameters(kappa, beta2, xi, phys.meandEdx(ke), pitch)
    return pdf


def theory_mpv(pdf):
    """Peak of the theoretical PDF, with the same scan as the maps (Hypfit::robust_max_x)."""
    try:
        return float(h_fit.robust_max_x(pdf, 0.0, 10.0, 10000))
    except Exception:
        xs = np.linspace(0.0, 10.0, 10000)
        return float(xs[int(np.argmax([pdf.Eval(x) for x in xs]))])


_THEORY_CACHE = {}


def theory_at(pdg, rr_center, pitch=SLICE_PITCH, x_min=-5.0, x_max=25.0, n_points=6001):
    """(MPV, x grid, PDF values on the grid, grid step), cached per residual range."""
    key = (pdg, round(float(rr_center), 4), pitch)
    if key not in _THEORY_CACHE:
        pdf = build_theoretical_pdf(pdg, rr_center, pitch)
        x_grid = np.linspace(x_min, x_max, n_points)
        pdf_vals = np.array([pdf.Eval(x) for x in x_grid])
        _THEORY_CACHE[key] = (theory_mpv(pdf), x_grid, pdf_vals, x_grid[1] - x_grid[0])
    return _THEORY_CACHE[key]


def gaussian_kernel(x, sigma):
    """Zero-mean Gaussian sampled on x (normalized by the caller)."""
    if sigma <= 0:
        kernel = np.zeros_like(x)
        kernel[np.argmin(np.abs(x))] = 1.0
        return kernel
    return np.exp(-0.5 * (x / sigma) ** 2)


def convolved_theoretical_pdf(x_query, sigma, shift, amplitude, x_grid, pdf_vals, dx):
    """amplitude * (PDF (x) unit-area Gaussian(sigma))(x - shift)."""
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return amplitude * np.interp(x_query - shift, x_grid, conv)


def fit_theoretical_conv_slice(bin_centers, bin_counts, bin_errs, x_grid, pdf_vals, dx,
                               mpv_theory, fit_range=None, sigma0=0.2,
                               sigma_bounds=(1e-4, 5.0), shift_bounds=(-3.0, 3.0)):
    """Fits (sigma, shift, amplitude) in [0.6, 1.4] x MPV; also returns chi2, ndf and
    the sigma-shift correlation. None if the fit fails or has fewer than 5 bins."""
    if fit_range is None:
        fit_range = (0.6 * mpv_theory, 1.4 * mpv_theory)
    mask = (bin_centers >= fit_range[0]) & (bin_centers <= fit_range[1]) & (bin_counts > 0)
    if mask.sum() < 5:
        return None
    xs, ys = bin_centers[mask], bin_counts[mask]
    yerr = np.where(bin_errs[mask] > 0, bin_errs[mask], 1.0)
    amp0 = ys.max() / (pdf_vals.max() if pdf_vals.max() > 0 else 1.0)

    def model(x, sigma, shift, amplitude):
        return convolved_theoretical_pdf(x, sigma, shift, amplitude, x_grid, pdf_vals, dx)

    try:
        popt, pcov = curve_fit(
            model, xs, ys, p0=[sigma0, 0.0, amp0], sigma=yerr, absolute_sigma=True,
            bounds=([sigma_bounds[0], shift_bounds[0], 0.0],
                    [sigma_bounds[1], shift_bounds[1], np.inf]),
            maxfev=20000,
        )
    except Exception:
        return None

    perr = np.sqrt(np.diag(pcov))
    chi2 = float(np.sum(((ys - model(xs, *popt)) / yerr) ** 2))
    denom = np.sqrt(pcov[0, 0] * pcov[1, 1])
    corr = pcov[0, 1] / denom if np.isfinite(denom) and denom > 0 else np.nan
    return dict(sigma=popt[0], sigma_err=perr[0], shift=popt[1], shift_err=perr[1],
                amplitude=popt[2], amplitude_err=perr[2], chi2=chi2,
                ndf=len(xs) - len(popt), corr_sigma_shift=corr, fit_range=fit_range)


def fit_rr_slices_theoretical(df, plane, tpc, pdg, rr_min=3.0, rr_max=40.0, rr_bin_width=1.0,
                              hist_nbins=150, hist_range=(0.0, 20.0), min_entries=30):
    """Slice fits of one (plane, tpc) hit DataFrame (columns rr, dedx). Keeps every
    converged fit with a finite shift error; the sigma_G quality cut is applied as a
    flag in run_slice_fits."""
    edges = np.arange(rr_min, rr_max + rr_bin_width, rr_bin_width)
    n_slices = len(edges) - 1
    rr = df["rr"].to_numpy(dtype=float)
    dedx = df["dedx"].to_numpy(dtype=float)
    ok = np.isfinite(rr) & np.isfinite(dedx) & (rr >= edges[0]) & (rr < edges[-1])

    # Sort the hits by residual-range slice once, instead of selecting each slice
    ibin = np.floor((rr[ok] - edges[0]) / rr_bin_width).astype(int)
    order = np.argsort(ibin, kind="stable")
    ibin, dedx_sorted = ibin[order], dedx[ok][order]
    bounds = np.searchsorted(ibin, np.arange(n_slices + 1))

    hist_edges = np.linspace(hist_range[0], hist_range[1], hist_nbins + 1)
    bin_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
    rows = []
    for i in range(n_slices):
        vals = dedx_sorted[bounds[i]:bounds[i + 1]]
        if len(vals) < min_entries:
            continue
        rr_center = 0.5 * (edges[i] + edges[i + 1])
        mpv_theory, x_grid, pdf_vals, dx = theory_at(pdg, rr_center)
        counts = np.histogram(vals, bins=hist_edges)[0].astype(float)
        errs = np.where(counts > 0, np.sqrt(counts), 1.0)
        fit = fit_theoretical_conv_slice(bin_centers, counts, errs, x_grid, pdf_vals, dx, mpv_theory)
        if fit is None or not np.isfinite(fit["shift_err"]):
            continue
        rows.append(dict(
            plane=plane, tpc=tpc, rr_center=rr_center, n_hits=len(vals),
            mpv_x=mpv_theory, mpv_theory=mpv_theory,
            gsigma=fit["sigma"], gsigma_err=fit["sigma_err"],
            shift=fit["shift"], shift_err=fit["shift_err"],
            chi2=fit["chi2"], ndf=fit["ndf"], corr_sigma_shift=fit["corr_sigma_shift"],
            fit_range_lo=fit["fit_range"][0], fit_range_hi=fit["fit_range"][1],
        ))
    return pd.DataFrame(rows)


def run_slice_fits(hit_dfs, pdg, rr_max):
    """Slice fits for planes 0-2 and TPC 0, 1 and both combined (tpc = -1).
    Every converged fit is kept; sigma_ok flags the slices whose sigma_G error lies
    in SLICE_SIGMA_ERR_RANGE. Returns {(plane, tpc): DataFrame}."""
    lo, hi = SLICE_SIGMA_ERR_RANGE
    results = {}
    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            sub = df if tpc == -1 else df[df["tpc"] == tpc]
            res = fit_rr_slices_theoretical(sub, plane, tpc, pdg, rr_max=rr_max)
            if len(res):
                e = res["gsigma_err"]
                res["sigma_ok"] = np.isfinite(e) & (e >= lo) & (e <= hi)
            n_ok = int(res["sigma_ok"].sum()) if len(res) else 0
            results[(plane, tpc)] = res
            print(f"  plane {plane}, TPC {'both' if tpc == -1 else tpc}: "
                  f"{len(res)} slices, {n_ok} with usable sigma_G")
    return results

In [ ]:
# ======================================================================
# Parametrizations of sigma_G and of the shift
# ======================================================================
def power_law(x, a, b, c):
    return a + b * np.power(x, c)


def exp_decay_plateau(rr, a, b, c):
    """a + b * exp(-rr / c)"""
    return a + b * np.exp(-rr / c)


def exp_decay_linear(rr, a, b, c, d):
    """a + b * exp(-rr / c) + d * rr"""
    return a + b * np.exp(-rr / c) + d * rr


def power_law_linear(rr, a, b, p, d):
    """a + b * rr^p + d * rr"""
    return a + b * np.power(rr, p) + d * rr


# Shift models: name -> (function, parameter names)
SHIFT_MODELS = {
    "exp_plateau": (exp_decay_plateau, ("a", "b", "c")),
    "exp_linear": (exp_decay_linear, ("a", "b", "c", "d")),
    "power_linear": (power_law_linear, ("a", "b", "p", "d")),
}


def eval_shift_fit(x, popt, model, rr_floor=None):
    """Shift model `model` at residual range x. With rr_floor, the model is evaluated
    at max(x, rr_floor), so it stays constant below rr_floor."""
    x = np.asarray(x, dtype=float)
    if rr_floor is not None:
        x = np.maximum(x, rr_floor)
    return SHIFT_MODELS[model][0](x, *popt)


def shift_fit_params(popt, perr, model):
    """{"model": model, name: value, name_err: error} for the shift-fit parameters."""
    out = {"model": model}
    for name, val, err in zip(SHIFT_MODELS[model][1], popt, perr):
        out[name], out[name + "_err"] = float(val), float(err)
    return out


def fit_sigma_vs_mpv(res, p0=(0.05, 0.1, 2.0),
                     bounds=((0.0, 0.0, -20.0), (2.0, 100.0, 20.0))):
    """sigma_G = a + b * MPV^c, MPV = theoretical MPV of the slice. Uses only the
    slices flagged sigma_ok by run_slice_fits. Returns (popt, perr)."""
    v = res.dropna(subset=["mpv_theory", "gsigma", "gsigma_err"])
    v = v[v["gsigma_err"] > 0]
    if "sigma_ok" in v:
        v = v[v["sigma_ok"]]
    if len(v) < 4:
        return None, None
    try:
        popt, pcov = curve_fit(power_law, v["mpv_theory"].to_numpy(), v["gsigma"].to_numpy(),
                               p0=p0, sigma=v["gsigma_err"].to_numpy(), absolute_sigma=True,
                               bounds=bounds, maxfev=20000)
    except Exception as exc:
        print(f"  sigma fit failed: {exc}")
        return None, None
    return popt, np.sqrt(np.diag(pcov))


def fit_shift_vs_rr(points, model="exp_linear", c_max=50.0, p_bounds=(-4.0, -0.05),
                    verbose=False):
    """Shift vs residual range. model: "exp_linear" (a, b, c, d), "exp_plateau" (a, b, c)
    or "power_linear" (a, b, p, d). c_max bounds the decay length, so the exponential
    stays the fast component; p_bounds bounds the power-law exponent."""
    if model not in SHIFT_MODELS:
        raise ValueError(f"Unknown shift model: {model}")
    func, names = SHIFT_MODELS[model]
    v = points.dropna(subset=["rr_center", "shift", "shift_err"])
    v = v[v["shift_err"] > 0].sort_values("rr_center")
    if len(v) < len(names) + 2:
        return None, None
    x, y, yerr = (v[c].to_numpy() for c in ("rr_center", "shift", "shift_err"))

    c0, p_start = 5.0, -1.0
    if model in ("exp_linear", "power_linear"):
        tail = x >= x.min() + (x.max() - x.min()) / 3.0
        d0, a0 = np.polyfit(x[tail], y[tail], 1, w=1.0 / yerr[tail])
        excess0 = y[0] - (a0 + d0 * x[0])
    if model == "exp_linear":
        p0 = [a0, excess0 * np.exp(x[0] / c0), c0, d0]
        bounds = ([-np.inf, -np.inf, 0.1, -np.inf], [np.inf, np.inf, c_max, np.inf])
    elif model == "power_linear":
        p0 = [a0, excess0 * np.power(x[0], -p_start), p_start, d0]
        bounds = ([-np.inf, -np.inf, p_bounds[0], -np.inf],
                  [np.inf, np.inf, p_bounds[1], np.inf])
    else:
        p0 = [y[-1], y[0] - y[-1], c0]
        bounds = ([-np.inf, -np.inf, 0.1], [np.inf, np.inf, c_max])

    try:
        popt, pcov = curve_fit(func, x, y, p0=p0, sigma=yerr, absolute_sigma=True,
                               bounds=bounds, maxfev=20000)
    except Exception as exc:
        print(f"  shift fit failed: {exc}")
        return None, None
    if verbose:
        chi2 = np.sum(((y - func(x, *popt)) / yerr) ** 2)
        print(f"  shift fit ({model}): chi2/ndf = {chi2:.1f}/{len(x) - len(popt)}")
    return popt, np.sqrt(np.diag(pcov))


# ---- Muon shifts transferred to pions at equal betagamma ----
def rr_to_betagamma(pdg, rr):
    """betagamma at residual range rr [cm]."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    return np.array([phys.KEtoMomentum(phys.KEFromRangeSpline(float(r))) / mass for r in rr])


def betagamma_to_rr(pdg, bg):
    """Residual range [cm] at which particle pdg has the given betagamma."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    return np.array([phys.RangeFromKESpline(phys.MomentumtoKE(float(b) * mass)) for b in bg])


def muon_points_as_pion(mu_res):
    """Muon shift points with rr_center replaced by the pion residual range of equal betagamma."""
    out = mu_res[["rr_center", "shift", "shift_err"]].dropna().copy()
    out["rr_center"] = betagamma_to_rr(211, rr_to_betagamma(13, out["rr_center"].to_numpy()))
    return out.sort_values("rr_center").reset_index(drop=True)


def combined_shift_points(pi_res, mu_res, rr_join=PION_RR_JOIN, overlap=PION_OVERLAP):
    """Pion shift points up to rr_join, then muon points at equal betagamma beyond it.
    The muon points are moved by the weighted mean pion - muon difference inside
    `overlap`. Returns (points, offset)."""
    pi = pi_res[["rr_center", "shift", "shift_err"]].dropna()
    mu = muon_points_as_pion(mu_res)
    ov = pi[(pi["rr_center"] >= overlap[0]) & (pi["rr_center"] <= overlap[1])]
    if len(ov) == 0 or len(mu) < 2:
        raise RuntimeError("No pion/muon overlap: check PION_OVERLAP and SLICE_RR_MAX")
    mu_at_pi = np.interp(ov["rr_center"].to_numpy(), mu["rr_center"].to_numpy(), mu["shift"].to_numpy())
    w = 1.0 / ov["shift_err"].to_numpy() ** 2
    offset = float(np.sum(w * (ov["shift"].to_numpy() - mu_at_pi)) / np.sum(w))
    mu_far = mu[mu["rr_center"] > rr_join].assign(shift=lambda d: d["shift"] + offset)
    points = pd.concat([pi[pi["rr_center"] <= rr_join], mu_far], ignore_index=True)
    return points.sort_values("rr_center").reset_index(drop=True), offset

In [ ]:
# ======================================================================
# Run the slice fits (muons first, so the target particle stays loaded)
# ======================================================================
SLICE_PARTICLES = [PARTICLE]
if PARTICLE == "pion" and PION_SHIFT_FROM_MUONS:
    SLICE_PARTICLES.append("muon")

slice_results = {}    # (particle, sample) -> {(plane, tpc): DataFrame}
for particle in reversed(SLICE_PARTICLES):
    pdg = {"pion": 211, "muon": 13}[particle]
    for sample in ("MC", "Data"):
        print(f"[slice fits] {particle} {sample}, rr < {SLICE_RR_MAX[particle]:g} cm")
        hits = get_sample(particle, sample)["hits"]
        slice_results[(particle, sample)] = run_slice_fits(hits, pdg, SLICE_RR_MAX[particle])
        if particle != PARTICLE:
            drop_sample(particle, sample)

In [ ]:
# ======================================================================
# Parametrizations for the target particle (fits, check plots, saved parameters)
# ======================================================================
import json
import os

SHIFT_MODEL = "power_linear"   # shift model, a key of SHIFT_MODELS

fit_params = {}     # sample -> {"sigma": {plane: popt}, "shift": {plane: popt}, "shift_model": name,
                    #            "rr_shift_min": cm, "rr_shift_max": cm}
shift_points = {}   # (sample, plane) -> points used for the shift fit

for sample in ("MC", "Data"):
    target = slice_results[(PARTICLE, sample)]
    sigma_par, shift_par, rr_start, rr_end = {}, {}, [], []
    for plane in range(3):
        res = target.get((plane, PARAM_TPC), pd.DataFrame())
        if not len(res):
            raise RuntimeError(f"No slice fits for {PARTICLE} {sample}, plane {plane}")
        sigma_par[plane], _ = fit_sigma_vs_mpv(res)

        if PARTICLE == "pion" and PION_SHIFT_FROM_MUONS:
            mu_res = slice_results[("muon", sample)].get((plane, PARAM_TPC), pd.DataFrame())
            points, offset = combined_shift_points(res, mu_res)
            print(f"[{sample}] plane {plane}: muon offset {offset:+.4f} MeV/cm")
        else:
            points = res[["rr_center", "shift", "shift_err"]].dropna()
        shift_points[(sample, plane)] = points
        print(f"[{sample}] plane {plane}: {len(points)} shift points up to "
              f"{points['rr_center'].max():.0f} cm")
        popt, perr = fit_shift_vs_rr(points, model=SHIFT_MODEL, verbose=True)
        shift_par[plane] = popt
        if popt is not None:
            pars = shift_fit_params(popt, perr, SHIFT_MODEL)
            print("    " + ", ".join(f"{n} = {pars[n]:.4g} +- {pars[n + '_err']:.2g}"
                                     for n in SHIFT_MODELS[SHIFT_MODEL][1]))
        rr_start.append(points["rr_center"].min())
        rr_end.append(points["rr_center"].max())
    fit_params[sample] = dict(sigma=sigma_par, shift=shift_par, shift_model=SHIFT_MODEL,
                              rr_shift_min=float(np.max(rr_start)),
                              rr_shift_max=float(np.min(rr_end)))

# ---- Muon parameters from the muon slice fits of this run (muon residual range, no offset) ----
mu_params = {}      # same layout as fit_params; empty unless the muon slices were fitted
mu_points = {}      # (sample, plane) -> muon points used for the muon shift fit

if PARTICLE == "pion" and PION_SHIFT_FROM_MUONS:
    for sample in ("MC", "Data"):
        sigma_par, shift_par, rr_start, rr_end = {}, {}, [], []
        for plane in range(3):
            mu_res = slice_results[("muon", sample)].get((plane, PARAM_TPC), pd.DataFrame())
            if not len(mu_res):
                raise RuntimeError(f"No slice fits for muon {sample}, plane {plane}")
            sigma_par[plane], _ = fit_sigma_vs_mpv(mu_res)
            points = mu_res[["rr_center", "shift", "shift_err"]].dropna()
            mu_points[(sample, plane)] = points
            print(f"[muon {sample}] plane {plane}: {len(points)} shift points up to "
                  f"{points['rr_center'].max():.0f} cm")
            shift_par[plane], _ = fit_shift_vs_rr(points, model=SHIFT_MODEL, verbose=True)
            rr_start.append(points["rr_center"].min())
            rr_end.append(points["rr_center"].max())
        mu_params[sample] = dict(sigma=sigma_par, shift=shift_par, shift_model=SHIFT_MODEL,
                                 rr_shift_min=float(np.max(rr_start)),
                                 rr_shift_max=float(np.min(rr_end)))


def plot_parametrizations(particle, params, points):
    """Shift vs residual range (top) and sigma_G vs MPV (bottom) of one particle: slice
    points and fits. params: sample -> fit parameters; points: (sample, plane) -> shift points."""
    colors = {"MC": "#d62728", "Data": "#1f77b4"}
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), layout="constrained")
    for plane in range(3):
        ax_s, ax_g = axes[0, plane], axes[1, plane]
        for sample, color in colors.items():
            pts = points[(sample, plane)]
            ax_s.errorbar(pts["rr_center"], pts["shift"], yerr=pts["shift_err"], fmt="o", ms=2,
                          color=color, alpha=0.5, elinewidth=0.6, label=f"{sample} slices")
            popt = params[sample]["shift"][plane]
            if popt is not None:
                xs = np.linspace(pts["rr_center"].min(), pts["rr_center"].max(), 400)
                ys = eval_shift_fit(xs, popt, params[sample]["shift_model"],
                                    rr_floor=params[sample]["rr_shift_min"])
                ax_s.plot(xs, ys, color=color, lw=2, label=f"{sample} fit")

            res = slice_results[(particle, sample)][(plane, PARAM_TPC)]
            if "sigma_ok" in res:
                res = res[res["sigma_ok"]]
            ax_g.errorbar(res["mpv_theory"], res["gsigma"], yerr=res["gsigma_err"], fmt="o", ms=2,
                          color=color, alpha=0.5, elinewidth=0.6, label=f"{sample} slices")
            popt = params[sample]["sigma"][plane]
            if popt is not None:
                xs = np.linspace(res["mpv_theory"].min(), res["mpv_theory"].max(), 400)
                ax_g.plot(xs, power_law(xs, *popt), color=color, lw=2, label=f"{sample} fit")

        if particle == "pion" and PION_SHIFT_FROM_MUONS:
            ax_s.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="Pion / muon points")
        ax_s.set_title(f"Plane {plane}", fontsize=16)
        ax_s.set_xlabel("Residual range [cm]", fontsize=14)
        ax_g.set_xlabel("Theoretical MPV [MeV/cm]", fontsize=14)
        for ax in (ax_s, ax_g):
            ax.grid(True, ls=":", alpha=0.5)
            ax.legend(fontsize=10)
    axes[0, 0].set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=14)
    axes[1, 0].set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=14)
    fig.suptitle(particle.capitalize(), fontsize=18)
    plt.show()


plot_parametrizations(PARTICLE, fit_params, shift_points)
if mu_params:
    plot_parametrizations("muon", mu_params, mu_points)

# ======================================================================
# Parameters of every particle fitted so far, as the PhysdEdx.cpp maps
# ======================================================================
CPP_PDGS = (13, 2212, 211)                  # order of the entries in each plane
CPP_PARAMS_FILE = "physdedx_params.json"    # fits of each particle, kept across runs


def _as_list(popt):
    return None if popt is None else [float(v) for v in popt]


def save_cpp_params(params_by_pdg, path=CPP_PARAMS_FILE):
    """Store the fits of each particle in params_by_pdg ({pdg: params}) in path, keeping the
    entries of the other particles. Returns everything stored."""
    stored = {}
    if os.path.exists(path):
        with open(path) as f:
            stored = json.load(f)
    for pdg, params in params_by_pdg.items():
        stored[str(pdg)] = {
            sample: {"sigma": [_as_list(fp["sigma"][plane]) for plane in range(3)],
                     "shift": [_as_list(fp["shift"][plane]) for plane in range(3)],
                     "shift_model": fp["shift_model"],
                     "rr_shift_min": fp["rr_shift_min"], "rr_shift_max": fp["rr_shift_max"]}
            for sample, fp in params.items()
        }
    with open(path, "w") as f:
        json.dump(stored, f, indent=1)
    return stored


def print_cpp_maps(stored):
    """Print the four PhysdEdx.cpp maps; particles without stored fits get -1 entries."""
    n_par = {"sigma": 3, "shift": len(SHIFT_MODELS[SHIFT_MODEL][1])}
    for sample, suffix in (("MC", ""), ("Data", "_data")):
        models = {pdg: stored[str(pdg)][sample]["shift_model"]
                  for pdg in CPP_PDGS if sample in stored.get(str(pdg), {})}
        if len(set(models.values())) > 1:
            print(f"// WARNING: {sample} shift models differ between particles: {models}")
        for key, name in (("sigma", "pdg_plane_map"), ("shift", "pdg_plane_shift_map")):
            print(f"vector<std::map<int, vector<double>>> PhysdEdx::{name}{suffix} = {{")
            for plane in range(3):
                rows = []
                for pdg in CPP_PDGS:
                    entry = stored.get(str(pdg), {}).get(sample)
                    popt = entry[key][plane] if entry is not None else None
                    vals = popt if popt is not None else [-1] * n_par[key]
                    rows.append(f"    {{{pdg}, {{" + ", ".join(f"{v:.8g}" for v in vals) + "}}")
                print(f"  // Plane {plane}")
                print("  {")
                print(",\n".join(rows))
                print("  }" + ("," if plane < 2 else ""))
            print("};\n")


# Save the target particle and, if fitted, the muons; then print the maps once
cpp_params = {PDG: fit_params}
if mu_params:
    cpp_params[13] = mu_params
print_cpp_maps(save_cpp_params(cpp_params))

In [ ]:
# ======================================================================
# Pion and muon fits vs betagamma: shift (top) and sigma_G (bottom), per plane
# Run after the parametrization cell (needs fit_params, mu_params, shift_points,
# mu_points, slice_results, rr_to_betagamma, eval_shift_fit, power_law).
# ======================================================================
from matplotlib.ticker import FixedFormatter, FixedLocator, FormatStrFormatter, NullFormatter, NullLocator

BG_TICKS = [0.3, 0.5, 0.7, 1, 1.5, 2, 3, 5, 7, 10]          # betagamma ticks (bottom axis)
RR_TICKS = [3, 5, 10, 20, 50, 100, 200, 300]                 # pion residual range ticks (top axis)


def _sigma_slices(particle, sample, plane):
    """Slice points used for the sigma_G fit, sorted by residual range."""
    res = slice_results[(particle, sample)][(plane, PARAM_TPC)]
    if "sigma_ok" in res:
        res = res[res["sigma_ok"]]
    return res.dropna(subset=["rr_center", "mpv_theory", "gsigma"]).sort_values("rr_center")


def plot_fits_vs_betagamma(show_points=True, n_grid=400):
    """Overlay of the pion and muon fits vs betagamma. Pion: solid, muon: dashed;
    MC: red, data: blue. The pion shift fit uses the pion slices below PION_RR_JOIN and
    the offset muon slices above it; the muon shift fit uses the muon slices only."""
    if not mu_params:
        raise RuntimeError("No muon fits (mu_params is empty): run with PARTICLE = 'pion' "
                           "and PION_SHIFT_FROM_MUONS = True")
    colors = {"MC": "#d62728", "Data": "#1f77b4"}
    particles = {"pion": (211, fit_params, shift_points, "-", "o"),
                 "muon": (13, mu_params, mu_points, "--", "s")}

    # Residual-range grids of the shift fits and their betagamma (independent of the plane)
    grids = {}
    for particle, (pdg, params, _, _, _) in particles.items():
        for sample in colors:
            p = params[sample]
            rr = np.linspace(p["rr_shift_min"], p["rr_shift_max"], n_grid)
            grids[(particle, sample)] = (rr, rr_to_betagamma(pdg, rr))
    bg_join = float(rr_to_betagamma(211, [PION_RR_JOIN])[0])

    fig, axes = plt.subplots(2, 3, figsize=(18, 11), layout="constrained")
    for plane in range(3):
        ax_s, ax_g = axes[0, plane], axes[1, plane]
        for particle, (pdg, params, points, ls, marker) in particles.items():
            for sample, color in colors.items():
                label = f"{particle.capitalize()} {sample}"

                # ---- shift ----
                if show_points:
                    pts = points[(sample, plane)]
                    if particle == "pion":    # pion slices only (the muon part is shown as muon)
                        pts = pts[pts["rr_center"] <= PION_RR_JOIN]
                    ax_s.errorbar(rr_to_betagamma(pdg, pts["rr_center"].to_numpy()), pts["shift"],
                                  yerr=pts["shift_err"], fmt=marker, ms=2, color=color,
                                  mfc=color if particle == "pion" else "none",
                                  alpha=0.35, elinewidth=0.5, label=f"{label} slices")
                popt = params[sample]["shift"][plane]
                if popt is not None:
                    rr, bg = grids[(particle, sample)]
                    ys = eval_shift_fit(rr, popt, params[sample]["shift_model"],
                                        rr_floor=params[sample]["rr_shift_min"])
                    ax_s.plot(bg, ys, color=color, ls=ls, lw=2.2, label=f"{label} fit")

                # ---- sigma_G ----
                res = _sigma_slices(particle, sample, plane)
                bg_res = rr_to_betagamma(pdg, res["rr_center"].to_numpy())
                if show_points:
                    ax_g.errorbar(bg_res, res["gsigma"], yerr=res["gsigma_err"], fmt=marker, ms=2,
                                  color=color, mfc=color if particle == "pion" else "none",
                                  alpha=0.35, elinewidth=0.5, label=f"{label} slices")
                popt = params[sample]["sigma"][plane]
                if popt is not None and len(res):
                    ax_g.plot(bg_res, power_law(res["mpv_theory"].to_numpy(), *popt),
                              color=color, ls=ls, lw=2.2, label=f"{label} fit")

        for ax in (ax_s, ax_g):
            ax.axvline(bg_join, color="gray", ls="-.", lw=1, label=f"Pion rr = {PION_RR_JOIN:g} cm")
            ax.set_xscale("log")
            ax.xaxis.set_major_locator(FixedLocator(BG_TICKS))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%g"))
            ax.xaxis.set_minor_formatter(NullFormatter())
            ax.set_xlabel(r"$\beta\gamma$", fontsize=15)
            ax.grid(True, which="major", ls=":", alpha=0.5)
            ax.tick_params(labelsize=12)

        # Pion residual range at the top of the shift panels (ticks placed at their betagamma)
        lo, hi = ax_s.get_xlim()
        rr_ticks = [r for r in RR_TICKS if lo <= rr_to_betagamma(211, [r])[0] <= hi]
        top = ax_s.twiny()
        top.set_xscale("log")
        top.set_xlim(lo, hi)
        top.xaxis.set_major_locator(FixedLocator(rr_to_betagamma(211, rr_ticks)))
        top.xaxis.set_major_formatter(FixedFormatter([f"{r:g}" for r in rr_ticks]))
        top.xaxis.set_minor_locator(NullLocator())
        top.set_xlabel("Pion residual range [cm]", fontsize=12)
        top.tick_params(labelsize=11)
        ax_s.set_title(f"Plane {plane}", fontsize=17, pad=8)

    axes[0, 0].set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=15)
    axes[1, 0].set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=15)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=5, fontsize=11)
    plt.show()


plot_fits_vs_betagamma()                     # fits and slice points
# plot_fits_vs_betagamma(show_points=False)  # fits only

In [ ]:
regions = [(0, 20), (20, 80), (80, 200), (200, 400)]
for (sample, plane), pts in sorted(shift_points.items()):
    popt = fit_params[sample]["shift"][plane]
    rr = pts["rr_center"].to_numpy()
    r = pts["shift"].to_numpy() - eval_shift_fit(rr, popt)
    parts = []
    for lo, hi in regions:
        sel = (rr >= lo) & (rr < hi)
        if sel.any():
            parts.append(f"{lo}-{hi} cm: rms {np.sqrt(np.mean(r[sel] ** 2)):.4f}, "
                         f"max {np.abs(r[sel]).max():.4f}")
    print(f"{sample:4s} plane {plane} | " + " | ".join(parts))

In [ ]:
regions = [(0, 20), (20, 80), (80, 200), (200, 400)]
for (sample, plane), pts in sorted(shift_points.items()):
    popt = fit_params[sample]["shift"][plane]
    rr = pts["rr_center"].to_numpy()
    err = pts["shift_err"].to_numpy()
    r = pts["shift"].to_numpy() - eval_shift_fit(rr, popt)
    parts = []
    for lo, hi in regions:
        sel = (rr >= lo) & (rr < hi)
        if sel.sum() < 2:
            continue
        w = 1.0 / err[sel] ** 2
        mean = np.sum(w * r[sel]) / np.sum(w)
        rms = np.sqrt(np.mean(r[sel] ** 2))
        stat = np.sqrt(np.mean(err[sel] ** 2))
        excess = np.sqrt(max(rms ** 2 - stat ** 2, 0.0))
        parts.append(f"{lo}-{hi}: mean {mean:+.4f}, stat {stat:.4f}, excess {excess:.4f}")
    print(f"{sample:4s} plane {plane} | " + " | ".join(parts))

In [ ]:
def overlap_prediction_test(pi_res, mu_res, fit_window=(40.0, 60.0), test_window=(60.0, 80.0)):
    """Offset from fit_window; muon prediction compared with the pion points in test_window.
    Returns (offset, mean deviation, chi2, number of test points)."""
    pi = pi_res[["rr_center", "shift", "shift_err"]].dropna()
    mu = muon_points_as_pion(mu_res)

    def mu_at(rr, col):
        return np.interp(rr, mu["rr_center"].to_numpy(), mu[col].to_numpy())

    a = pi[(pi["rr_center"] >= fit_window[0]) & (pi["rr_center"] < fit_window[1])]
    w = 1.0 / a["shift_err"] ** 2
    offset = float(np.sum(w * (a["shift"] - mu_at(a["rr_center"], "shift"))) / np.sum(w))

    b = pi[(pi["rr_center"] >= test_window[0]) & (pi["rr_center"] < test_window[1])]
    dev = b["shift"] - (mu_at(b["rr_center"], "shift") + offset)
    err = np.hypot(b["shift_err"], mu_at(b["rr_center"], "shift_err"))
    return offset, float(np.mean(dev)), float(np.sum((dev / err) ** 2)), len(b)


for sample in ("MC", "Data"):
    for plane in range(3):
        off, mean_dev, chi2, n = overlap_prediction_test(
            slice_results[(PARTICLE, sample)][(plane, PARAM_TPC)],
            slice_results[("muon", sample)][(plane, PARAM_TPC)])
        print(f"{sample:4s} plane {plane}: offset {off:+.4f}, mean deviation at 60-80 cm "
              f"{mean_dev:+.4f} MeV/cm, chi2/n = {chi2:.1f}/{n}")

In [ ]:
def data_minus_mc(particle, plane, tpc=-1):
    mc, da = (slice_results[(particle, s)][(plane, tpc)].set_index("rr_center") for s in ("MC", "Data"))
    j = da.join(mc[["shift", "shift_err"]], rsuffix="_mc", how="inner")
    return j["mpv_theory"], j["shift"] - j["shift_mc"], np.hypot(j["shift_err"], j["shift_err_mc"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for plane, ax in enumerate(axes):
    for particle, color in (("pion", "tab:red"), ("muon", "tab:blue")):
        ax.errorbar(*data_minus_mc(particle, plane), fmt=".", color=color, label=particle)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set(xlabel="Theoretical MPV [MeV/cm]", title=f"Plane {plane}")
axes[0].set(ylabel="Shift, data - MC [MeV/cm]")
axes[0].legend()